# PBB-style JEPA on CIFAR10-DVS — standalone

Direct port of the PBB v2 recipe to event-camera data. Same `pbb_core` library as `pbb_cifar10.ipynb`. The whole folder is copy-paste portable — no `omnibird/` dependency.

**Differences vs. the CIFAR-10 variant:**
- Coord dim 3 (x, y, t) instead of 2 (y, x).
- Signal dim 2 (one-hot polarity) instead of 3 (RGB).
- 8192-event pool instead of 410-pixel pool.
- Pool size + multi-block sizes scaled accordingly.

**Caveat:** this is the cleanest possible JEPA baseline for events, but per-event tokens carry just 1 bit of polarity (vs 24 bits of RGB per pixel). The PBB recipe's "direct gather + target centering + per-token LN" defenses are closer to their limit here. If this run probes weakly, switch to either:
- `cifar10_dvs_temporal.ipynb` in the parent `notebooks/` (causal time-axis JEPA), or
- `cifar10_dvs_eventmae.ipynb` (MAE reconstruction — guaranteed collapse-free since target is data).


## 1. Setup

In [ ]:
import os, sys, math, time, copy, ssl, glob
sys.path.insert(0, os.path.abspath('.'))   # so pbb_core.py is importable
ssl._create_default_https_context = ssl._create_unverified_context

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
import matplotlib.pyplot as plt

from pbb_core import (
    PBBEncoder, PBBPredictor,
    TargetCenter, ema_update, make_momentum_schedule,
    jepa_loss, gather_target_features,
    precompute_grid_orderings, short_params,
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0); np.random.seed(0)
N_GPUS = min(4, torch.cuda.device_count())
USE_DP = (DEVICE == "cuda") and (N_GPUS > 1)
print(f"GPUs visible = {torch.cuda.device_count()}  using = {N_GPUS}  DataParallel = {USE_DP}")

DATA_ROOT  = "../data/cifar10_dvs_omnibird"
TRAIN_FRAC = 0.8

# ── Config ───────────────────────────────────────────────────────────────
SIDE       = 64                                # grid resolution per axis for SFC ranks
K_POOL     = 8192                              # events per clip (cap+pad)
K_CTX      = 2048                              # context events
N_PRED     = 4                                 # target blocks
K_TGT_PER_BLOCK = 512                          # events per target block
N_TGT      = N_PRED * K_TGT_PER_BLOCK          # 2048 target events

D_MODEL      = 256
N_LAYERS_ENC = 6
N_HEADS      = 8
DIM_HEAD     = 32
BLOCK_SIZE   = 8
WINDOW       = 1
N_RANDOM     = 2
N_GLOBAL     = 2
FOURIER_DIM  = 96
FOURIER_SCALE = 15.0

D_PRED        = 192
N_LAYERS_PRED = 4
N_HEADS_PRED  = 6
DIM_HEAD_PRED = 32

EMA_START = 0.999
EMA_END   = 1.0

EPOCHS         = 200
BATCH_SIZE     = 32
LR             = 2e-4
WEIGHT_DECAY   = 0.0
WARMUP_EPOCHS  = 5
PROBE_INTERVAL = 10
PROBE_EPOCHS   = 2
LOG_EVERY      = 25
CKPT_DIR       = "./checkpoints_pbb_cifar10_dvs"
os.makedirs(CKPT_DIR, exist_ok=True)
print(f"K_pool={K_POOL}  K_ctx={K_CTX}  N_tgt={N_TGT}  ({N_PRED}x{K_TGT_PER_BLOCK})")


## 2. CIFAR10-DVS dataset

In [ ]:
class CIFAR10DVSFromClips:
    def __init__(self, root, sensor_hw=(128, 128)):
        self.root = root; self.h, self.w = sensor_hw
        self.clips = sorted(glob.glob(os.path.join(root, "clip_*")))
        if not self.clips:
            raise RuntimeError(f"No clip_* in {root} (resolved: {os.path.abspath(root)})")
    def __len__(self): return len(self.clips)
    def __getitem__(self, idx):
        clip = self.clips[idx]
        ev = np.load(os.path.join(clip, "events_0.npy")).astype(np.float32)
        if ev.shape[0] == 0: ev = np.zeros((1, 4), dtype=np.float32)
        x = ev[:, 0] / max(self.w - 1, 1) * 2.0 - 1.0
        y = ev[:, 1] / max(self.h - 1, 1) * 2.0 - 1.0
        t_raw = ev[:, 2]
        t = (t_raw - t_raw.min()) / max(t_raw.max() - t_raw.min(), 1.0) * 2.0 - 1.0
        p = ev[:, 3]
        if p.max() <= 1.0 and p.min() >= 0.0: p = p * 2.0 - 1.0
        out = np.stack([x, y, t, p], axis=1).astype(np.float32)
        with open(os.path.join(clip, "label_0.txt")) as f: label = int(f.read().strip())
        return out, label

GRID_RANKS = {k: v for k, v in precompute_grid_orderings(SIDE, ndim=3).items()}


def quantize_coords_3d(coords, side=SIDE, value_range=(-1.0, 1.0)):
    lo, hi = value_range
    norm = (coords - lo) / (hi - lo)
    norm = norm.clamp(0.0, 1.0)
    cell = (norm * (side - 1)).round().long().clamp(0, side - 1)
    return cell[..., 2] * side * side + cell[..., 1] * side + cell[..., 0]


class JEPAChunkDVS(Dataset):
    def __init__(self, base, train=True):
        self.base = base
        self.train = train

    def __len__(self): return len(self.base)

    def _knn_block(self, pool_xyt, candidate_local, K, anchor_local, exclude_mask):
        d2 = ((pool_xyt - pool_xyt[anchor_local]) ** 2).sum(-1)
        d2[~candidate_local] = float("inf")
        d2[exclude_mask]     = float("inf")
        topk = torch.topk(d2, K, largest=False).indices
        return topk

    def _signal(self, polarity):
        # one-hot: col 0 = ON, col 1 = OFF
        sig = torch.zeros(polarity.shape[0], 2, dtype=torch.float32)
        sig[polarity > 0, 0] = 1.0
        sig[polarity < 0, 1] = 1.0
        return sig

    def __getitem__(self, idx):
        events, label = self.base[idx]
        events = torch.as_tensor(events, dtype=torch.float32)
        N_raw = events.shape[0]
        # Cap / pad to K_POOL
        if N_raw > K_POOL:
            sel_rng = np.random.RandomState(idx + 7919) if not self.train else np.random
            sel = sel_rng.choice(N_raw, K_POOL, replace=False)
            sel.sort()
            events = events[sel]
        if events.shape[0] < K_POOL:
            pad = torch.zeros(K_POOL - events.shape[0], 4, dtype=events.dtype)
            events = torch.cat([events, pad])
        N_real = min(N_raw, K_POOL)
        kpm = torch.zeros(K_POOL, dtype=torch.bool)
        kpm[N_real:] = True

        # Sort by Hilbert curve (3D)
        cell_ids = quantize_coords_3d(events[:, :3])
        ranks = GRID_RANKS["hilbert"][cell_ids]
        # Push padded events to tail
        ranks = ranks + kpm.long() * (K_POOL + 1)
        order = ranks.argsort(stable=True)
        events = events[order]
        kpm    = kpm[order]

        pool_coords = events[:, :3]                          # (K_pool, 3)
        pool_signal = self._signal(events[:, 3])             # (K_pool, 2)

        candidate = ~kpm                                      # only real events
        exclude   = torch.zeros(K_POOL, dtype=torch.bool)

        if self.train:
            rng = np.random.RandomState()
            tgt_blocks = []
            for _ in range(N_PRED):
                allowed = (candidate & ~exclude).nonzero(as_tuple=False).squeeze(-1)
                if len(allowed) < K_TGT_PER_BLOCK:
                    break
                anchor = allowed[rng.randint(len(allowed))].item()
                blk = self._knn_block(pool_coords, candidate, K_TGT_PER_BLOCK,
                                       anchor, exclude)
                tgt_blocks.append(blk)
                exclude[blk] = True
            tgt_pool_pos = torch.cat(tgt_blocks) if tgt_blocks else torch.zeros(0, dtype=torch.long)
            expected = N_PRED * K_TGT_PER_BLOCK
            if len(tgt_pool_pos) < expected:
                pad = torch.full((expected - len(tgt_pool_pos),),
                                  tgt_pool_pos[-1].item() if len(tgt_pool_pos) else 0,
                                  dtype=torch.long)
                tgt_pool_pos = torch.cat([tgt_pool_pos, pad])

            remaining = (candidate & ~exclude).nonzero(as_tuple=False).squeeze(-1)
            if len(remaining) >= K_CTX:
                anchor = remaining[rng.randint(len(remaining))].item()
                ctx_pool_pos = self._knn_block(pool_coords, candidate, K_CTX, anchor, exclude)
            else:
                ctx_pool_pos = remaining
                if len(ctx_pool_pos) < K_CTX:
                    pad = torch.full((K_CTX - len(ctx_pool_pos),),
                                      ctx_pool_pos[0].item() if len(ctx_pool_pos) else 0,
                                      dtype=torch.long)
                    ctx_pool_pos = torch.cat([ctx_pool_pos, pad])

            sample = {
                "pool_coords":   pool_coords.contiguous(),
                "pool_signal":   pool_signal.contiguous(),
                "pool_kpm":      kpm.contiguous(),
                "tgt_pool_pos":  tgt_pool_pos.contiguous(),
                "ctx_pool_pos":  ctx_pool_pos.contiguous(),
                "label":         int(label),
            }
        else:
            real_idx = candidate.nonzero(as_tuple=False).squeeze(-1)
            ctx_pool_pos = real_idx[:K_CTX] if len(real_idx) >= K_CTX else \
                            torch.cat([real_idx, torch.zeros(K_CTX - len(real_idx), dtype=torch.long)])
            sample = {
                "pool_coords":   pool_coords.contiguous(),
                "pool_signal":   pool_signal.contiguous(),
                "pool_kpm":      kpm.contiguous(),
                "ctx_pool_pos":  ctx_pool_pos.contiguous(),
                "label":         int(label),
            }
        return {k: v.clone() if isinstance(v, torch.Tensor) else v for k, v in sample.items()}


base = CIFAR10DVSFromClips(DATA_ROOT)
n = len(base); rng = np.random.RandomState(0)
perm = rng.permutation(n); n_train = int(n * TRAIN_FRAC)
train_idx, test_idx = perm[:n_train], perm[n_train:]
class _Subset:
    def __init__(self, b, i): self.b=b; self.i=i
    def __len__(self): return len(self.i)
    def __getitem__(self, k): return self.b[int(self.i[k])]
base_train = _Subset(base, train_idx); base_test = _Subset(base, test_idx)

train_ds      = JEPAChunkDVS(base_train, train=True)
train_eval_ds = JEPAChunkDVS(base_train, train=False)
test_ds       = JEPAChunkDVS(base_test,  train=False)
train_loader      = DataLoader(train_ds,      batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
train_eval_loader = DataLoader(train_eval_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
test_loader       = DataLoader(test_ds,       batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
print(f"train={len(train_ds)}  test={len(test_ds)}")


## 3. Models

In [ ]:
context_encoder = PBBEncoder(
    d_model=D_MODEL, n_layers=N_LAYERS_ENC,
    n_heads=N_HEADS, dim_head=DIM_HEAD,
    block_size=BLOCK_SIZE, window=WINDOW,
    n_random=N_RANDOM, n_global=N_GLOBAL,
    signal_dim=2, coord_dim=3,
    fourier_dim=FOURIER_DIM, fourier_scale=FOURIER_SCALE,
).to(DEVICE)
target_encoder = copy.deepcopy(context_encoder).to(DEVICE)
for p in target_encoder.parameters(): p.requires_grad_(False)

predictor = PBBPredictor(
    d_model=D_MODEL, d_pred=D_PRED,
    n_layers=N_LAYERS_PRED, n_heads=N_HEADS_PRED, dim_head=DIM_HEAD_PRED,
    coord_dim=3, fourier_dim=FOURIER_DIM, fourier_scale=FOURIER_SCALE,
    pos_symmetric=True,
).to(DEVICE)
target_center = TargetCenter(D_MODEL, momentum=0.9).to(DEVICE)

if USE_DP:
    device_ids = list(range(N_GPUS))
    context_encoder = nn.DataParallel(context_encoder, device_ids=device_ids)
    target_encoder  = nn.DataParallel(target_encoder,  device_ids=device_ids)
    predictor       = nn.DataParallel(predictor,       device_ids=device_ids)

def _unwrap(m): return m.module if isinstance(m, nn.DataParallel) else m
print(f"context_encoder: {short_params(_unwrap(context_encoder))}")
print(f"predictor      : {short_params(_unwrap(predictor))}")


## 4. SFC orderings per batch

In [ ]:
def orderings_from_coords_3d(coords):
    grid_ranks = {k: v.to(coords.device) for k, v in GRID_RANKS.items()}
    cell_ids = quantize_coords_3d(coords)
    out = {}
    for name, full in grid_ranks.items():
        ranks = full[cell_ids]
        perm = ranks.argsort(dim=-1)
        inv  = torch.empty_like(perm)
        arange = torch.arange(perm.shape[1], device=perm.device).unsqueeze(0).expand_as(perm)
        inv.scatter_(1, perm, arange)
        out[name] = {"perm": perm, "inverse": inv}
    return out


## 5. Optim + resume

In [ ]:
params = list(_unwrap(context_encoder).parameters()) + list(_unwrap(predictor).parameters())
optimizer = AdamW(params, lr=LR, weight_decay=WEIGHT_DECAY)
total_steps = EPOCHS * len(train_loader)
warmup_steps = WARMUP_EPOCHS * len(train_loader)
def lr_lambda(step):
    if step < warmup_steps:
        return step / max(warmup_steps, 1)
    p = (step - warmup_steps) / max(total_steps - warmup_steps, 1)
    return 0.5 * (1 + math.cos(math.pi * p))
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
momentum_gen = make_momentum_schedule(EMA_START, EMA_END, total_steps)

LAST = os.path.join(CKPT_DIR, "pbb_dvs_last.pt")
BEST = os.path.join(CKPT_DIR, "pbb_dvs_best.pt")
RESUME = True
history = {"loss": [], "diag_log": [], "diag_steps": [], "probe_accs": []}
start_epoch, best_loss, global_step = 1, float("inf"), 0
m = EMA_START
if RESUME and os.path.exists(LAST):
    s = torch.load(LAST, map_location=DEVICE, weights_only=False)
    _unwrap(context_encoder).load_state_dict(s["context_encoder"])
    _unwrap(target_encoder).load_state_dict(s["target_encoder"])
    _unwrap(predictor).load_state_dict(s["predictor"])
    target_center.load_state_dict(s["center"])
    optimizer.load_state_dict(s["opt"]); scheduler.load_state_dict(s["sched"])
    history = s.get("history", history)
    global_step = s.get("global_step", 0); best_loss = s.get("best_loss", float("inf"))
    start_epoch = s["epoch"] + 1
    for _ in range(global_step):
        try: m = next(momentum_gen)
        except StopIteration: m = EMA_END
    print(f"resumed @ ep {s['epoch']}, step {global_step}")
else:
    print("starting fresh.")


## 6. Probe

In [ ]:
class LinearProbe(nn.Module):
    def __init__(self, d, n=10):
        super().__init__()
        self.fc = nn.Linear(d, n)
    def forward(self, z): return self.fc(z)


def _ctx_features(batch, enc):
    pool_coords = batch["pool_coords"].to(DEVICE)
    pool_signal = batch["pool_signal"].to(DEVICE)
    ctx_pool_pos = batch["ctx_pool_pos"].to(DEVICE)
    pool_kpm     = batch["pool_kpm"].to(DEVICE)
    pool_ords = orderings_from_coords_3d(pool_coords)
    with torch.no_grad():
        g_pool = enc(pool_signal, pool_coords, pool_ords, key_padding_mask=pool_kpm)
    B, K_pool, D = g_pool.shape
    idx = ctx_pool_pos.unsqueeze(-1).expand(B, ctx_pool_pos.size(1), D)
    g_ctx = torch.gather(g_pool, 1, idx)
    return g_ctx.mean(dim=1)


def quick_probe(num_epochs=PROBE_EPOCHS, lr=1e-3, wd=1e-4):
    enc = _unwrap(context_encoder)
    saved_rg = [p.requires_grad for p in enc.parameters()]
    for p in enc.parameters(): p.requires_grad_(False)
    enc.eval()
    probe = LinearProbe(D_MODEL, 10).to(DEVICE)
    opt = AdamW(probe.parameters(), lr=lr, weight_decay=wd)
    ce  = nn.CrossEntropyLoss()
    for _ in range(num_epochs):
        probe.train()
        for b in train_eval_loader:
            y = b["label"].to(DEVICE)
            opt.zero_grad(set_to_none=True)
            ce(probe(_ctx_features(b, enc)), y).backward()
            opt.step()
    probe.eval()
    correct = total = 0
    with torch.no_grad():
        for b in test_loader:
            y = b["label"].to(DEVICE)
            correct += (probe(_ctx_features(b, enc)).argmax(-1) == y).sum().item()
            total += y.size(0)
    for p, rg in zip(enc.parameters(), saved_rg): p.requires_grad_(rg)
    return correct / max(total, 1)


## 7. Training loop

In [ ]:
epoch = start_epoch - 1
try:
    for epoch in range(start_epoch, EPOCHS + 1):
        context_encoder.train(); predictor.train()
        epoch_loss, steps = 0.0, 0
        t0 = time.time()
        for batch in train_loader:
            pool_coords = batch["pool_coords"].to(DEVICE)
            pool_signal = batch["pool_signal"].to(DEVICE)
            pool_kpm    = batch["pool_kpm"].to(DEVICE)
            tgt_pool_pos = batch["tgt_pool_pos"].to(DEVICE)
            ctx_pool_pos = batch["ctx_pool_pos"].to(DEVICE)
            pool_ords = orderings_from_coords_3d(pool_coords)

            with torch.no_grad():
                g_tgt = target_encoder(pool_signal, pool_coords, pool_ords,
                                        key_padding_mask=pool_kpm)
                h_tgt_raw = gather_target_features(g_tgt, tgt_pool_pos)
                target_center.update(h_tgt_raw)
                h_tgt = F.layer_norm(target_center(h_tgt_raw), (h_tgt_raw.size(-1),))

            B, _, _ = pool_signal.shape
            ctx_signal = torch.gather(pool_signal, 1,
                                       ctx_pool_pos.unsqueeze(-1).expand(B, ctx_pool_pos.size(1), 2))
            ctx_coords = torch.gather(pool_coords, 1,
                                       ctx_pool_pos.unsqueeze(-1).expand(B, ctx_pool_pos.size(1), 3))
            ctx_ords = orderings_from_coords_3d(ctx_coords)
            g_ctx = context_encoder(ctx_signal, ctx_coords, ctx_ords)

            tgt_coords = torch.gather(pool_coords, 1,
                                       tgt_pool_pos.unsqueeze(-1).expand(B, tgt_pool_pos.size(1), 3))
            h_pred = predictor(g_ctx, tgt_coords, ctx_coords=ctx_coords)

            loss = jepa_loss(h_pred, h_tgt, loss_type="smooth_l1")
            optimizer.zero_grad(set_to_none=True); loss.backward()
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            optimizer.step(); scheduler.step()

            try: m = next(momentum_gen)
            except StopIteration: m = EMA_END
            ema_update(_unwrap(target_encoder), _unwrap(context_encoder), m)

            global_step += 1; epoch_loss += loss.item(); steps += 1
            if global_step % LOG_EVERY == 0:
                pred_std = h_pred.std(dim=0).mean().item()
                tgt_std  = h_tgt.std(dim=0).mean().item()
                cos = F.cosine_similarity(h_pred, h_tgt, dim=-1).mean().item()
                print(f"[ep{epoch:03d} st{global_step:06d}]  loss={loss.item():.4f}  "
                      f"pred_std={pred_std:.3f}  tgt_std={tgt_std:.3f}  cos={cos:.3f}  "
                      f"lr={scheduler.get_last_lr()[0]:.1e}  m={m:.5f}")
                history["diag_steps"].append(global_step)
                history["diag_log"].append({"pred_std": pred_std, "tgt_std": tgt_std,
                                              "cos": cos, "loss": loss.item()})

        avg = epoch_loss / max(steps, 1)
        history["loss"].append(avg)
        improved = avg < best_loss
        if improved: best_loss = avg
        state = {
            "epoch": epoch,
            "context_encoder": _unwrap(context_encoder).state_dict(),
            "target_encoder":  _unwrap(target_encoder).state_dict(),
            "predictor":       _unwrap(predictor).state_dict(),
            "center":          target_center.state_dict(),
            "opt": optimizer.state_dict(), "sched": scheduler.state_dict(),
            "global_step": global_step, "best_loss": best_loss, "history": history,
        }
        torch.save(state, LAST)
        if improved: torch.save(state, BEST)
        print(f"=== ep {epoch:03d}/{EPOCHS}  loss={avg:.4f}  m={m:.5f}  "
              f"{time.time()-t0:.1f}s" + ("  *" if improved else ""))
        if PROBE_INTERVAL > 0 and epoch % PROBE_INTERVAL == 0:
            tp = time.time()
            acc = quick_probe()
            history["probe_accs"].append((epoch, acc))
            print(f"  [probe ep {epoch:03d}]  test_acc = {acc:.4f}  ({time.time()-tp:.1f}s)")
    print("\nDone.")
except KeyboardInterrupt:
    print(f"\nInterrupted at epoch {epoch}.  Checkpoints in {CKPT_DIR}.")


## 8. Final long-form probe

In [ ]:
LOAD_BEST = True
n_probe_epochs = 30
probe_lr = 1e-3; probe_wd = 1e-4

if LOAD_BEST and os.path.exists(BEST):
    s = torch.load(BEST, map_location=DEVICE, weights_only=False)
    _unwrap(context_encoder).load_state_dict(s["context_encoder"])
    print(f"loaded best @ ep {s['epoch']}")
enc = _unwrap(context_encoder); enc.eval()
for p in enc.parameters(): p.requires_grad_(False)
probe = LinearProbe(D_MODEL, 10).to(DEVICE)
opt = AdamW(probe.parameters(), lr=probe_lr, weight_decay=probe_wd)
ce = nn.CrossEntropyLoss()
history_probe = {"train_acc": [], "test_acc": []}
for ep in range(1, n_probe_epochs + 1):
    probe.train()
    c = t = 0
    for b in train_eval_loader:
        y = b["label"].to(DEVICE)
        logits = probe(_ctx_features(b, enc))
        opt.zero_grad(set_to_none=True); ce(logits, y).backward(); opt.step()
        c += (logits.argmax(-1) == y).sum().item(); t += y.size(0)
    ta = c / t
    probe.eval()
    c2 = t2 = 0
    with torch.no_grad():
        for b in test_loader:
            y = b["label"].to(DEVICE)
            c2 += (probe(_ctx_features(b, enc)).argmax(-1) == y).sum().item(); t2 += y.size(0)
    te = c2 / t2
    history_probe["train_acc"].append(ta); history_probe["test_acc"].append(te)
    print(f"  ep {ep:02d}  train={ta:.4f}  test={te:.4f}")
best_test = max(history_probe["test_acc"])
print(f"\nbest probe test = {best_test:.4f}")
